In [4]:
import cupy as cp
import cupyx

In [5]:
def get_im2col_idx(x_shape: tuple, kernel_size: int, pad: int, stride: int):
    N, C, H, W = x_shape
    H_out = (H + 2 * pad - kernel_size) // stride + 1
    W_out = (W + 2 * pad - kernel_size) // stride + 1

    i0 = cp.repeat(cp.arange(kernel_size), kernel_size)
    i0 = cp.tile(i0, C)
    i1 = stride * cp.repeat(cp.arange(H_out), W_out)
    i = i0.reshape(-1, 1) + i1.reshape(1, -1)

    j0 = cp.tile(cp.arange(kernel_size), kernel_size)
    j0 = cp.tile(j0, C)
    j1 = stride * cp.tile(cp.arange(W_out), H_out)
    j = j0.reshape(-1, 1) + j1.reshape(1, -1)

    k = cp.repeat(cp.arange(C), kernel_size * kernel_size).reshape(-1, 1)
    return k, i, j


def im2col(x: cp.ndarray, kernel_size: int, pad: int, stride: int) -> cp.ndarray:
    N, C, H, W = x.shape
    x_pad = cp.pad(x, ((0, 0), (0, 0), (pad, pad),
                   (pad, pad)), mode='constant')
    k, i, j = get_im2col_idx(x.shape, kernel_size, pad, stride)
    cols = x_pad[:, k, i, j]  # N, C*k*k, H_out * W_out
    cols = cols.transpose(1, 2, 0).reshape(C * kernel_size * kernel_size, -1)
    return cols


def col2im(cols: cp.ndarray, x_shape: tuple, kernel_size: int, pad: int, stride: int) -> cp.ndarray:
    N, C, H, W = x_shape

    H_padded = H + 2 * pad
    W_padded = W + 2 * pad

    x_padded = cp.zeros((N, C, H_padded, W_padded), dtype=cols.dtype)

    k, i, j = get_im2col_idx(x_shape, kernel_size, pad, stride)

    # cols: (C*K*K, N*H_out*W_out)
    cols_reshaped = cols.reshape(C * kernel_size * kernel_size, -1, N)
    cols_reshaped = cols_reshaped.transpose(
        2, 0, 1)   # (N, C*K*K, H_out*W_out)

    n_idx = cp.arange(N)[:, None, None]   # (N,1,1)

    cupyx.scatter_add(
        x_padded, (n_idx, k[None, :, :], i[None, :, :], j[None, :, :]), cols_reshaped)

    if pad == 0:
        return x_padded
    return x_padded[:, :, pad:pad + H, pad:pad + W]


class Conv2D:
    """2D Convolutional layer with support for forward and backward pass

    Parameters:
    -----------
    in_channels: int
      Number of channels in the input image
    out_channels: int
      Number of channels produced by the convolution
    kernel_size: int
      Size of the convolutional kernel.
    stride: int
      Stride of the convolution. Default: 1.
    padding: int
      Padding added to all four sides of the input. Default: 0
    bias: bool
      Whether to include a bias term. Default: False
    dtype: cp.dtype
      Data type of the weights and biases. Default: cp.float64
    """

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0,
                 bias=False, dtype=cp.float64):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.dtype = dtype

        self.weights = cp.zeros(
            (out_channels, in_channels, kernel_size, kernel_size), dtype=dtype)
        self.bias = None
        if bias:
            self.bias = cp.zeros(out_channels, dtype=dtype)

        self.init()

        self.db = None
        self.dW = None
        self.x = None
        self.cols = None
        self.out_shape = None

    def init(self):
        fan_in = self.in_channels * self.kernel_size * self.kernel_size
        fan_out = self.out_channels * self.kernel_size * self.kernel_size

        limit = cp.sqrt(6.0 / (fan_in + fan_out))

        self.weights = cp.random.uniform(
            low=-limit,
            high=limit,
            size=self.weights.shape
        ).astype(self.dtype)

        if self.bias is not None:
            self.bias = cp.zeros_like(self.bias)

    def set_weights(self, weights: cp.ndarray):
        expected_shape = (self.out_channels, self.in_channels,
                          self.kernel_size, self.kernel_size)
        assert weights.shape == expected_shape, \
            f"Expected weights shape {expected_shape}, got {weights.shape}"
        self.weights = weights.astype(self.dtype, copy=False)

    def set_bias(self, bias: cp.ndarray):
        expected_shape = (self.out_channels,)
        assert bias.shape == expected_shape, \
            f"Expected bias shape {expected_shape}, got {bias.shape}"
        self.bias = bias.astype(self.dtype, copy=False)

    def forward(self, x: cp.ndarray) -> cp.ndarray:
        """
        Parameters
        ----------
        x : cp.ndarray of shape (N, C_in, H, W)

        Returns
        -------
        cp.ndarray of shape (N, C_out, H_out, W_out)
        """
        assert x.ndim == 4, f"Expected 4D input (N, C, H, W), got {x.ndim}D"

        N, C, H, W = x.shape
        assert C == self.in_channels, f"Input has {C} channels, expected {self.in_channels}"

        p = self.padding
        s = self.stride
        k = self.kernel_size

        assert (H + 2 * p - k) % s == 0, f"Height of input is not divisible by stride"
        assert (W + 2 * p - k) % s == 0, f"Width of input is not divisible by stride"
        H_out = (H + 2 * p - k) // s + 1
        W_out = (W + 2 * p - k) // s + 1

        cols = im2col(x, k, pad=p, stride=s)
        weight_2d = self.weights.reshape(self.out_channels, -1)
        out_2d = weight_2d @ cols

        out = out_2d.reshape(self.out_channels, H_out,
                             W_out, N).transpose(3, 0, 1, 2)
        if self.bias is not None:
            out += self.bias[None, :, None, None]

        self.x = x
        self.cols = cols
        self.out_shape = out.shape

        return out

    def backward(self, grad_output: cp.ndarray):
        assert self.x is not None, "Run forward() before backward()"
        assert grad_output.shape == self.out_shape, \
            f"Expected grad_output shape {self.out_shape}, got {grad_output.shape}"

        N, C_out, H_out, W_out = self.out_shape

        # bias gradient
        if self.bias is not None:
            self.db = grad_output.sum(axis=(0, 2, 3))

        grad_output_2d = grad_output.transpose(1, 2, 3, 0).reshape(C_out, -1)

        # weight gradient
        dW_2d = grad_output_2d @ self.cols.T
        self.dW = dW_2d.reshape(
            self.out_channels,
            self.in_channels,
            self.kernel_size,
            self.kernel_size
        )

        # input gradient
        weight_2d = self.weights.reshape(self.out_channels, -1)
        dcols = weight_2d.T @ grad_output_2d
        dx = col2im(dcols, self.x.shape, self.kernel_size,
                    self.padding, self.stride)

        return dx

In [6]:
# --- Manual forward test ---
cp.random.seed(42)

conv = Conv2D(in_channels=1, out_channels=1,
              kernel_size=2, stride=1, padding=0)

x = cp.array([[
    [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]]
]], dtype=cp.float32)   # shape (1, 1, 3, 3)

w = cp.array([[
    [[1, 0],
     [0, -1]]
]], dtype=cp.float32)   # shape (1, 1, 2, 2)

b = cp.array([1.2], dtype=cp.float32)

conv.set_weights(w)
conv.set_bias(b)

out = conv.forward(x)

expected = cp.array([[
    [[-4, -4],
     [-4, -4]]
]], dtype=cp.float32) + 1.2

print("Input shape:   ", x.shape)
print("Weights shape: ", w.shape)
print("Output shape:  ", out.shape)
print("Output:\n", out)

assert cp.allclose(
    out, expected), f"Forward mismatch:\n{out}\nvs expected:\n{expected}"
print("Manual forward test passed.\n")

Input shape:    (1, 1, 3, 3)
Weights shape:  (1, 1, 2, 2)
Output shape:   (1, 1, 2, 2)
Output:
 [[[[-2.79999995 -2.79999995]
   [-2.79999995 -2.79999995]]]]
Manual forward test passed.



In [7]:
# --- Forward test against PyTorch ---

import cupy as cp
import numpy as np
import torch
import torch.nn.functional as F

cp.random.seed(42)
torch.manual_seed(42)

# Small deterministic test
N, C_in, H, W = 2, 3, 5, 5
C_out = 4
kernel_size = 3
stride = 1
padding = 1

x_cp = cp.random.randn(N, C_in, H, W).astype(cp.float32)
w_cp = cp.random.randn(C_out, C_in, kernel_size,
                       kernel_size).astype(cp.float32)
b_cp = cp.random.randn(C_out).astype(cp.float32)

conv = Conv2D(
    in_channels=C_in,
    out_channels=C_out,
    kernel_size=kernel_size,
    stride=stride,
    padding=padding,
    dtype=cp.float32
)
conv.set_weights(w_cp)
conv.set_bias(b_cp)

out_cp = conv.forward(x_cp)

# Convert to PyTorch
x_torch = torch.from_numpy(cp.asnumpy(x_cp))
w_torch = torch.from_numpy(cp.asnumpy(w_cp))
b_torch = torch.from_numpy(cp.asnumpy(b_cp))

out_torch = F.conv2d(
    x_torch,
    w_torch,
    bias=b_torch,
    stride=stride,
    padding=padding
)

out_ours = cp.asnumpy(out_cp)
out_ref = out_torch.detach().cpu().numpy()

max_diff = np.max(np.abs(out_ours - out_ref))
print("Our output shape:   ", out_ours.shape)
print("PyTorch output shape:", out_ref.shape)
print("Max abs diff:", max_diff)

assert out_ours.shape == out_ref.shape, "Shape mismatch"
assert np.allclose(out_ours, out_ref,
                   atol=1e-5), f"Mismatch with PyTorch, max diff = {max_diff}"

print("Forward test vs PyTorch passed.")

Our output shape:    (2, 4, 5, 5)
PyTorch output shape: (2, 4, 5, 5)
Max abs diff: 1.9073486e-06
Forward test vs PyTorch passed.


In [13]:
# --- Backward test against PyTorch ---
import cupy as cp
import numpy as np
import torch
import torch.nn.functional as F

cp.random.seed(42)
torch.manual_seed(42)


def compare_arrays(name, a_cp, b_torch, atol=1e-5):
    a = cp.asnumpy(a_cp)
    b = b_torch.detach().cpu().numpy()
    max_diff = np.max(np.abs(a - b))
    print(f"{name}: shape={a.shape}, max_abs_diff={max_diff}")
    assert a.shape == b.shape, f"{name} shape mismatch: {a.shape} vs {b.shape}"
    assert np.allclose(
        a, b, atol=atol), f"{name} mismatch: max diff = {max_diff}"


configs = [
    # N, C_in, H, W, C_out, K, stride, padding, bias
    (1, 1, 4, 4, 1, 2, 1, 0, True),
    (2, 3, 5, 5, 4, 3, 1, 1, True),
    (2, 3, 7, 7, 5, 3, 2, 1, True),
    (1, 2, 8, 8, 3, 5, 1, 2, False),
    (2, 1, 6, 6, 2, 1, 1, 0, True),
    (64, 3, 448, 448, 6, 3, 1, 2, False)
]

for idx, (N, C_in, H, W, C_out, K, stride, padding, use_bias) in enumerate(configs):
    cp.random.seed(100 + idx)
    torch.manual_seed(100 + idx)

    x_cp = cp.random.randn(N, C_in, H, W).astype(cp.float64)
    w_cp = cp.random.randn(C_out, C_in, K, K).astype(cp.float64)
    b_cp = cp.random.randn(C_out).astype(cp.float64) if use_bias else None

    conv = Conv2D(
        in_channels=C_in,
        out_channels=C_out,
        kernel_size=K,
        stride=stride,
        padding=padding,
        bias=use_bias,
        dtype=cp.float64
    )
    conv.set_weights(w_cp)
    if use_bias:
        conv.set_bias(b_cp)

    out_cp = conv.forward(x_cp)

    x_torch = torch.tensor(cp.asnumpy(
        x_cp), dtype=torch.float64, requires_grad=True)
    w_torch = torch.tensor(cp.asnumpy(
        w_cp), dtype=torch.float64, requires_grad=True)
    b_torch = None
    if use_bias:
        b_torch = torch.tensor(cp.asnumpy(
            b_cp), dtype=torch.float64, requires_grad=True)

    out_torch = F.conv2d(
        x_torch,
        w_torch,
        bias=b_torch,
        stride=stride,
        padding=padding
    )

    compare_arrays(f"forward cfg {idx}", out_cp, out_torch, atol=1e-8)

    grad_out_cp = cp.random.randn(*out_cp.shape).astype(cp.float64)
    grad_out_torch = torch.tensor(cp.asnumpy(grad_out_cp), dtype=torch.float64)

    dx_cp = conv.backward(grad_out_cp)
    out_torch.backward(grad_out_torch)

    compare_arrays(f"dx cfg {idx}", dx_cp, x_torch.grad, atol=1e-8)
    compare_arrays(f"dW cfg {idx}", conv.dW, w_torch.grad, atol=1e-8)

    if use_bias:
        compare_arrays(f"db cfg {idx}", conv.db, b_torch.grad, atol=1e-8)

    print(f"Config {idx} passed.\n")

print("All backward config tests passed.")

forward cfg 0: shape=(1, 1, 3, 3), max_abs_diff=4.440892098500626e-16
dx cfg 0: shape=(1, 1, 4, 4), max_abs_diff=0.0
dW cfg 0: shape=(1, 1, 2, 2), max_abs_diff=8.881784197001252e-16
db cfg 0: shape=(1,), max_abs_diff=1.1102230246251565e-16
Config 0 passed.

forward cfg 1: shape=(2, 4, 5, 5), max_abs_diff=3.552713678800501e-15
dx cfg 1: shape=(2, 3, 5, 5), max_abs_diff=1.7763568394002505e-15
dW cfg 1: shape=(4, 3, 3, 3), max_abs_diff=1.0658141036401503e-14
db cfg 1: shape=(4,), max_abs_diff=1.7763568394002505e-15
Config 1 passed.

forward cfg 2: shape=(2, 5, 4, 4), max_abs_diff=3.552713678800501e-15
dx cfg 2: shape=(2, 3, 7, 7), max_abs_diff=1.7763568394002505e-15
dW cfg 2: shape=(5, 3, 3, 3), max_abs_diff=2.6645352591003757e-15
db cfg 2: shape=(5,), max_abs_diff=1.3322676295501878e-15
Config 2 passed.

forward cfg 3: shape=(1, 3, 8, 8), max_abs_diff=8.881784197001252e-15
dx cfg 3: shape=(1, 2, 8, 8), max_abs_diff=1.0658141036401503e-14
dW cfg 3: shape=(3, 2, 5, 5), max_abs_diff=3.55271